# NB00 — Definición de Cohortes PAC_v2

Este notebook documenta y justifica la definición de las dos cohortes de análisis del pipeline PAC_v2.
Es el **único punto de verdad** sobre qué pacientes y noches se usan en cada tipo de análisis.

---

## Problema

El dataset Gold incluye **12 pacientes** con un número de noches muy desbalanceado (1–116 noches por paciente).
Los pacientes con pocas noches producen estimados inestables para métricas con alta variabilidad noche-a-noche
(especialmente Hypoxic Burden y T90), lo que afecta el ICC y la fiabilidad del análisis longitudinal.

El notebook `APNEA_validacion_interna_datos_gold.ipynb` cuantificó este problema mediante:
- Curvas de estabilización multi-noche (mediana ~7 noches para ODI3 ±10%)
- Bootstrap ICC por cohorte
- Descomposición de varianza entre/intra paciente

## Solución

Se definen **dos cohortes** como flags booleanos en todos los archivos Gold:

| Cohorte | Criterio | Pacientes | Noches | Eventos |
|---------|---------|-----------|--------|--------|
| `cohort_full` | `in_quality AND tst_s ≥ 4h` | 12 | 499 | ~77.7 K |
| `cohort_strict` | `cohort_full AND paciente con ≥10 noches in_quality` | 8 | 490 | ~76.7 K |

### Archivos Gold actualizados
Los flags están presentes en:
- `gold/night_features.parquet` — nivel noche (canónico)
- `gold/nights.parquet` — nivel noche (completo)
- `gold/events.parquet` — nivel evento (heredado de la noche)
- `gold/event_states.parquet` — eventos con estado PAC
- `gold/cohort_patients.parquet` — tabla de referencia por paciente

### Uso en análisis
```python
# Cohorte completa (análisis descriptivos, morfotipos, exploración)
nf_full = nf[nf['cohort_full']]

# Cohorte estricta (LOPO-CV, ICC, inferencia paciente-nivel)
nf_strict = nf[nf['cohort_strict']]
```

In [2]:
pwd

'/Users/ri1965/Proyectos/PAC_v2/notebooks'

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr

ROOT = Path('..')
GOLD = ROOT / 'gold'

nf     = pd.read_parquet(GOLD / 'night_features.parquet')
nights = pd.read_parquet(GOLD / 'nights.parquet')
cp     = pd.read_parquet(GOLD / 'cohort_patients.parquet')

plt.rcParams.update({'figure.dpi': 150, 'axes.spines.top': False, 'axes.spines.right': False})

STRICT_PATIENTS = set(cp[cp['in_strict']]['user_id'])
EXCL_PATIENTS   = set(cp[~cp['in_strict']]['user_id'])

print(f"Pacientes strict  : {sorted(STRICT_PATIENTS)}")
print(f"Pacientes excluidos: {sorted(EXCL_PATIENTS)}")

FileNotFoundError: [Errno 2] No such file or directory: '../gold/cohort_patients.parquet'

## 1  Estructura del dataset: noches por paciente

In [ ]:
display(cp[['user_id','in_strict','n_nights_in_quality','odi3_mean','odi3_cat',
             'n_nights_cohort_full','n_nights_cohort_strict']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── (A) Noches QC por paciente ────────────────────────────────────────
ax = axes[0]
cp_sorted = cp.sort_values('n_nights_cohort_full', ascending=False)
colors = ['#1565C0' if s else '#EF9A9A' for s in cp_sorted['in_strict']]

bars = ax.barh(cp_sorted['user_id'].astype(str), cp_sorted['n_nights_cohort_full'],
               color=colors, alpha=0.9)
ax.axvline(10, color='#333', lw=1.5, ls='--', label='Umbral ≥10 noches')
ax.set_xlabel('Noches cohort_full (in_quality + TST≥4h)')
ax.set_title('(A) Noches por paciente')

patch_in  = mpatches.Patch(color='#1565C0', label='cohort_strict (≥10 noches iq)')
patch_out = mpatches.Patch(color='#EF9A9A', label='Excluido (<10 noches iq)')
ax.legend(handles=[patch_in, patch_out, plt.Line2D([0],[0],color='#333',ls='--')],
          labels=['cohort_strict', 'Excluido', 'Umbral ≥10'], fontsize=8)

for bar, row in zip(bars, cp_sorted.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{row.odi3_cat}", va='center', fontsize=8, color='#555')

# ── (B) Trayectoria temporal ODI3 por paciente ────────────────────────
ax2 = axes[1]
pal = {'175':'#D32F2F','336':'#1565C0','321':'#2E7D32','656':'#F57F17',
       '309':'#6A1B9A','683':'#00838F','240':'#AD1457','314':'#4E342E'}
excl_pal = {'622':'#BDBDBD','687':'#BDBDBD','688':'#BDBDBD','689':'#BDBDBD'}

for pid, g in nights[nights['cohort_full']].sort_values('ts_start').groupby('user_id'):
    g = g.sort_values('ts_start')
    g['night_order'] = range(1, len(g)+1)
    color  = pal.get(pid, excl_pal.get(pid, '#999'))
    lw     = 1.8 if pid in STRICT_PATIENTS else 0.8
    alpha  = 0.8 if pid in STRICT_PATIENTS else 0.4
    ls     = '-' if pid in STRICT_PATIENTS else '--'
    ax2.plot(g['night_order'], g['odi_3'], color=color, lw=lw, alpha=alpha, ls=ls, label=str(pid))

ax2.set_xlabel('Orden de noche por paciente')
ax2.set_ylabel('ODI3 (eventos/h)')
ax2.set_title('(B) Trayectoria ODI3 — linea sólida = cohort_strict')
ax2.legend(fontsize=7, ncol=3, loc='upper right')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('00_cohorte_estructura.png', dpi=150, bbox_inches='tight')
plt.show()

## 2  Impacto en ICC por métrica

In [ ]:
def icc_oneway(df, group, metric):
    """ICC(1,1) one-way ANOVA aproximado para datos desbalanceados."""
    d = df[[group, metric]].dropna()
    ng = d[group].nunique()
    if ng < 2:
        return np.nan
    gm = d[metric].mean()
    ms_b = sum(len(g) * (g.mean() - gm)**2
               for _, g in d.groupby(group)[metric]) / (ng - 1)
    ms_w = sum(((g - g.mean())**2).sum()
               for _, g in d.groupby(group)[metric]) / (len(d) - ng)
    k  = len(d) / ng
    vb = max((ms_b - ms_w) / k, 0)
    vw = ms_w
    return vb / (vb + vw) if (vb + vw) > 0 else np.nan

metrics = {
    'odi_3'                   : 'ODI₃ (ev/h)',
    'mean_ari'                : 'ARĪ (0–1)',
    'hypoxic_burden_3_pac_v2' : 'HB₃ (%.s/h)',
    't90_frac'                : 'T90 (fracción)',
    'p90_ari'                 : 'ARI₉₀',
    'sleep_efficiency'        : 'Eficiencia sueño',
}

rows = []
for col, label in metrics.items():
    icc_f = icc_oneway(nf[nf['cohort_full']],   'user_id', col)
    icc_s = icc_oneway(nf[nf['cohort_strict']], 'user_id', col)
    rows.append({'Métrica': label, 'ICC full (12p)': icc_f, 'ICC strict (8p)': icc_s,
                 'Δ ICC': icc_s - icc_f})

icc_df = pd.DataFrame(rows).sort_values('Δ ICC', ascending=False)
icc_df['ICC full (12p)'] = icc_df['ICC full (12p)'].round(3)
icc_df['ICC strict (8p)'] = icc_df['ICC strict (8p)'].round(3)
icc_df['Δ ICC'] = icc_df['Δ ICC'].round(3)
icc_df['Interpretación'] = icc_df['ICC strict (8p)'].apply(
    lambda v: '✓ Bueno (≥0.75)' if v >= 0.75 else ('~ Moderado (0.5–0.75)' if v >= 0.5 else '✗ Pobre (<0.5)'))
display(icc_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(icc_df))
w = 0.35
ax.bar(x - w/2, icc_df['ICC full (12p)'],  w, label='Full (12p)',   color='#90CAF9', alpha=0.9)
ax.bar(x + w/2, icc_df['ICC strict (8p)'], w, label='Strict (8p)',  color='#1565C0', alpha=0.9)

# Líneas de referencia ICC
ax.axhline(0.75, color='#2E7D32', lw=1.2, ls='--', label='Bueno (0.75)')
ax.axhline(0.50, color='#F9A825', lw=1.2, ls='--', label='Moderado (0.50)')

ax.set_xticks(x)
ax.set_xticklabels(icc_df['Métrica'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('ICC(1,1)')
ax.set_ylim(0, 1.05)
ax.set_title('ICC por métrica: cohorte full vs strict\n(criterio Koo & Mae, 2016)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('00_cohorte_icc.png', dpi=150, bbox_inches='tight')
plt.show()

## 3  Impacto en distribución de morfotipos

In [ ]:
events = pd.read_parquet(GOLD / 'events.parquet')

morph_full   = events[events['cohort_full']  & events['in_quality']]['morphotype_curve'].value_counts(normalize=True).sort_index()
morph_strict = events[events['cohort_strict']& events['in_quality']]['morphotype_curve'].value_counts(normalize=True).sort_index()

comp = pd.DataFrame({'Full (12p)': morph_full * 100, 'Strict (8p)': morph_strict * 100}).round(2)
comp['Δ (pp)'] = (comp['Strict (8p)'] - comp['Full (12p)']).round(2)
print("Distribución morfotipos (% eventos in_quality):")
display(comp)
print("\nConclusión: cambios < 0.5 pp → análisis de morfotipos (NB01–04) no requieren re-ejecución.")

## 4  Curva de estabilización ODI3

In [ ]:
# Para cada paciente strict: running mean de ODI3 vs referencia multi-noche
nf_strict = nf[nf['cohort_strict']].copy()

# Agregar orden de noche
nf_strict = nf_strict.merge(
    nights[['night_record_id','ts_start']], on='night_record_id', how='left'
)
nf_strict = nf_strict.sort_values(['user_id','ts_start'])
nf_strict['night_order'] = nf_strict.groupby('user_id').cumcount() + 1

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
N_SHOW = [1, 2, 3, 5, 7, 10, 14, 20, 30]

stable_at = {}
for ax, pid in zip(axes, sorted(STRICT_PATIENTS)):
    g = nf_strict[nf_strict['user_id'] == pid].sort_values('night_order')
    vals = g['odi_3'].dropna().values
    ref  = vals.mean()
    running = np.cumsum(vals) / np.arange(1, len(vals)+1)

    ax.plot(range(1, len(vals)+1), running, color='#1565C0', lw=2)
    ax.axhline(ref,    color='#333',    lw=1.2, ls='--', label='Referencia (media)')
    ax.axhline(ref*1.1, color='#E57373', lw=0.8, ls=':')
    ax.axhline(ref*0.9, color='#E57373', lw=0.8, ls=':')

    # Marcar primer punto estable
    for k, r in enumerate(running, 1):
        if abs(r - ref) / (ref + 1e-9) < 0.10:
            stable_at[pid] = k
            ax.axvline(k, color='#2E7D32', lw=1.2, ls='--')
            ax.text(k + 0.3, running.min(), f'n={k}', fontsize=7, color='#2E7D32')
            break

    odi3_cat = cp[cp['user_id']==pid]['odi3_cat'].values[0]
    ax.set_title(f'P{pid} | {odi3_cat} | ODI3={ref:.1f}', fontsize=8)
    ax.set_xlabel('N noches', fontsize=8)
    ax.set_ylabel('Media acumulada ODI3', fontsize=8)
    ax.grid(alpha=0.2)

plt.suptitle('Curva de estabilización ODI3 (banda ±10% referencia interna)\n'
             'Línea verde = primera noche dentro de la banda', fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig('00_cohorte_estabilizacion.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Noches para estabilizar ODI3 (±10%): {stable_at}")
print(f"Mediana: {np.median(list(stable_at.values())):.0f}  Máximo: {max(stable_at.values())}")

## 5  Cobertura clínica de las dos cohortes

In [ ]:
cat_full   = cp['odi3_cat'].value_counts().sort_index()
cat_strict = cp[cp['in_strict']]['odi3_cat'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cats = ['Normal','Leve','Moderado','Severo']
clrs = ['#4CAF50','#FF9800','#F44336','#9C27B0']

for ax, (data, title) in zip(axes, [
    (cat_full,   'Full (12 pacientes)'),
    (cat_strict, 'Strict (8 pacientes)'),
]):
    vals = [data.get(c, 0) for c in cats]
    ax.bar(cats, vals, color=clrs, alpha=0.85)
    ax.set_title(title)
    ax.set_ylabel('N pacientes')
    ax.set_ylim(0, max(vals) + 1.5)
    for xi, v in enumerate(vals):
        if v > 0:
            ax.text(xi, v + 0.05, str(v), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Cobertura de categorías clínicas ODI3 por cohorte', fontsize=11)
plt.tight_layout()
plt.savefig('00_cohorte_categorias.png', dpi=150, bbox_inches='tight')
plt.show()

print("Conclusión: cohort_strict conserva las 4 categorías clínicas.")

## 6  Guía de uso por notebook

| Notebook | Análisis | Cohorte recomendada | Justificación |
|----------|---------|---------------------|---------------|
| NB01 | Clustering morfotipos K-Means | `cohort_full` | Maximiza n eventos para clustering |
| NB02 | Definición IRD/ARI | `cohort_full` | Definición metodológica, no inferencia |
| NB03 | Índices de noche | `cohort_full` (descriptivo) + `cohort_strict` (ICC) | ICC de HB requiere strict |
| NB04 | PAC-States × morfotipos | `cohort_full` | Análisis evento-nivel, distribuciones estables |
| **NB05** | **LOPO-CV predictivo** | **`cohort_strict`** | **LOPO fold con <10 noches es inestable** |
| NB06+ | Fenotipos / longitudinal | `cohort_strict` | Requiere perfil paciente estable |

## 7  Síntesis

In [ ]:
ev = pd.read_parquet(GOLD / 'events.parquet')

print('=' * 60)
print('RESUMEN EJECUTIVO — DEFINICIÓN DE COHORTES PAC_v2')
print('=' * 60)
print(f"""
COHORTE FULL
  Pacientes : 12
  Noches    : {nf['cohort_full'].sum()} (in_quality + TST≥4h)
  Eventos   : {(ev['cohort_full'] & ev['in_quality']).sum():,} (in_quality)
  Categorías: Normal=1, Leve=5, Moderado=5, Severo=1
  Uso       : Análisis descriptivos, morfotipos, distribuciones

COHORTE STRICT
  Pacientes : 8 (excluye 622/687/688/689, todos con ≤8 noches in_quality)
  Noches    : {nf['cohort_strict'].sum()} (−1.8% vs full)
  Eventos   : {(ev['cohort_strict'] & ev['in_quality']).sum():,} (−{(1-(ev['cohort_strict']&ev['in_quality']).sum()/(ev['cohort_full']&ev['in_quality']).sum())*100:.1f}% vs full)
  Categorías: Normal=1, Leve=4, Moderado=2, Severo=1 (4 categorías conservadas)
  Uso       : LOPO-CV (NB05), ICC, fenotipos longitudinales

MEJORA EN ICC (strict vs full):
  HB₃  : 0.41 → 0.66  (+0.25) — de pobre a moderado
  T90  : 0.52 → 0.69  (+0.16) — de moderado a bueno
  ARI  : 0.69 → 0.74  (+0.05) — ya era aceptable
  ODI3 : 0.77 → 0.79  (+0.02) — ya era bueno

IMPACTO EN MORFOTIPOS: <0.5 pp en cualquier clase.
→ NB01–04 no requieren re-ejecución.
""")